In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path


In [5]:
df = pd.read_pickle(Path(os.getcwd()).parent / 'datos' / 'intermedios' / 'tablon_analitico.pkl')

In [ ]:
# Extraer componentes de fecha y hora
df['mes'] = df['fecha_hora'].dt.month
df['dia'] = df['fecha_hora'].dt.day
df['hora'] = df['fecha_hora'].dt.hour
df['minuto'] = df['fecha_hora'].dt.minute
df['time'] = df['fecha_hora'].dt.strftime('%H:%M:%S')


Nuevas variables creadas:
- mes: [5 6]
- dia: [15 16 17 18 19 20 21 22 23 24]
- hora: [0 1 2 3 4 5 6 7 8 9]
- minuto: [ 0 15 30 45]
- time (primeros 10): ['00:00:00' '00:15:00' '00:30:00' '00:45:00' '01:00:00' '01:15:00'
 '01:30:00' '01:45:00' '02:00:00' '02:15:00']


In [14]:
# pasar fecha_hora a índice
df.set_index('fecha_hora', inplace=True)

In [20]:
def eficiencia_inverter(dc,ac):
    temp = np.where(dc > 0, ac / dc, 1) * 100
    return np.where(temp > 100, 100, temp)

In [21]:
df['eficiencia'] = eficiencia_inverter(df['potencia_dc_kw'], df['potencia_ac_kw'])

In [24]:
orden = ['id_planta','id_inversor','id_sensor_meteorologico','mes','dia','time','hora','minuto','irradiacion_wh_m2','temperatura_ambiente_c','temperatura_modulo_c','potencia_dc_kw','eficiencia','potencia_ac_kw','energia_diaria_kwh','energia_total_kwh']

In [25]:
df = df[orden]

In [30]:
df_dia = df.groupby(['id_planta','id_inversor']).resample('D').agg({
    'irradiacion_wh_m2': ['min','mean','max','sum'],
    'temperatura_ambiente_c': ['min','mean','max'],
    'temperatura_modulo_c': ['min','mean','max'],
    'potencia_dc_kw': ['min','mean','max'],
    'eficiencia': ['min','mean','max',],
    'potencia_ac_kw': ['min','mean','max',],
    'energia_diaria_kwh': ['min','mean','max'],
    'energia_total_kwh': ['min','mean','max']
})

In [33]:
# Aplanar las columnas del MultiIndex
# Aplanar las columnas del MultiIndex
df_dia.columns = ['_'.join(col) for col in df_dia.columns]

In [35]:
# Resetear el índice para convertir id_planta e id_inversor en columnas
df_dia = df_dia.reset_index()

In [36]:
# Establecer fecha_hora como índice
df_dia = df_dia.set_index('fecha_hora')

In [38]:
df.to_pickle(Path(os.getcwd()).parent / 'datos' / 'intermedios' / 'tablon_analitico_preparado.pkl')
df_dia.to_pickle(Path(os.getcwd()).parent / 'datos' / 'intermedios' / 'tablon_analitico_diario.pkl')